In [1]:
import pandas as pd
import requests

iphones = {
    "iPhone 12": 699,
    "iPhone 13": 799,
    "iPhone 14": 899,
    "iPhone 15": 999,
    "iPhone 15 Pro": 1199
}
prices_df = pd.DataFrame(list(iphones.items()), columns=["Product", "Price"])


def fetch_iphone_info(models):
    url = "https://en.wikipedia.org/w/api.php"
    def fetch(model):
        res = requests.get(url, params={
            "action": "query", "format": "json", "titles": model,
            "prop": "extracts|pageimages", "exintro": True, "explaintext": True, "pithumbsize": 500
        }).json()
        page = next(iter(res.get("query", {}).get("pages", {}).values()), {})
        return page.get("extract", "No info."), page.get("thumbnail", {}).get("source", "No image.")
    return {m: fetch(m) for m in models}

info = fetch_iphone_info(["iPhone 12", "iPhone 13", "iPhone 14", "iPhone 15", "iPhone 15 Pro"])
info_df = pd.DataFrame([(name, desc, img) for name, (desc, img) in info.items()], columns=["Product", "Description", "Image_URL"])


def fetch_news_cleaned():
    url = "https://en.wikipedia.org/w/api.php"
    params = {"action": "query", "format": "json", "list": "search", "srsearch": "iPhone", "srlimit": 10}
    response = requests.get(url, params=params)
    data = response.json().get("query", {}).get("search", [])
    return [{"title": item["title"], "description": item["snippet"], "link": f"https://en.wikipedia.org/?curid={item['pageid']}"} for item in data]

news = fetch_news_cleaned()
news_df = pd.DataFrame(news)


combined_df = prices_df.merge(info_df, on="Product", how="left")


combined_df["News_Title"] = news_df["title"][0] if not news_df.empty else "No news"
combined_df["News_Description"] = news_df["description"][0] if not news_df.empty else "No description"
combined_df["News_Link"] = news_df["link"][0] if not news_df.empty else "No link"


combined_df.to_csv("products_data.csv", index=False)
print("The basic data is saved in a file. products_data.csv")

ModuleNotFoundError: No module named 'pandas'

In [13]:
import pandas as pd


df = pd.read_csv("products_data.csv")
print("Original Data Before Adding Region:")
print(df)



regions = ["North America", "Europe", "Asia", "North America", "Europe"]
df["Region"] = regions


df.to_csv("products_data.csv", index=False)
print("Updated Data After Adding Region:")
print(df)

Original Data Before Adding Region:
         Product  Price                                        Description  \
0      iPhone 12    699  The iPhone 12 and iPhone 12 Mini (stylized and...   
1      iPhone 13    799  The iPhone 13 and iPhone 13 Mini (stylized as ...   
2      iPhone 14    899  The iPhone 14 and iPhone 14 Plus are smartphon...   
3      iPhone 15    999  The iPhone 15 and iPhone 15 Plus are smartphon...   
4  iPhone 15 Pro   1199  The iPhone 15 Pro and iPhone 15 Pro Max are sm...   

                                           Image_URL News_Title  \
0  https://upload.wikimedia.org/wikipedia/commons...     IPhone   
1  https://upload.wikimedia.org/wikipedia/commons...     IPhone   
2  https://upload.wikimedia.org/wikipedia/commons...     IPhone   
3  https://upload.wikimedia.org/wikipedia/commons...     IPhone   
4  https://upload.wikimedia.org/wikipedia/commons...     IPhone   

                                    News_Description  \
0  accessibility features. Up to the

In [ ]:
import pandas as pd
import networkx as nx


iphones = {
    "iPhone 12": 699,
    "iPhone 13": 799,
    "iPhone 14": 899,
    "iPhone 15": 999,
    "iPhone 15 Pro": 1199
}

G = nx.Graph()
for model, price in iphones.items():
    G.add_node(model, price=price)
models = list(iphones.keys())
for i in range(len(models)):
    for j in range(i + 1, len(models)):
        diff = abs(iphones[models[i]] - iphones[models[j]])
        if diff <= 300:
            G.add_edge(models[i], models[j], weight=diff)


edges = [(u, v, d['weight']) for u, v, d in G.edges(data=True)]
relationships_df = pd.DataFrame(edges, columns=["Product_A", "Product_B", "Frequency"])


relationships_df.to_csv("relationships_data.csv", index=False)
print("The relationship data is saved in a file. relationships_data.csv")

تم حفظ داتا العلاقات في ملف relationships_data.csv


In [1]:
#Network Analysis with NetworkX:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from community import community_louvain


relationships_df = pd.read_csv("relationships_data.csv")
print("Relationships Data:")
print(relationships_df)


G = nx.Graph()
for _, row in relationships_df.iterrows():
    G.add_edge(row["Product_A"], row["Product_B"], weight=row["Frequency"])


plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G)
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=500, font_size=10, edge_color='gray')
plt.title("Product Network (Relationships Based on Price Difference)")
plt.show()


degree_centrality = nx.degree_centrality(G)
print("Importance of Products (Popularity):", degree_centrality)


partition = community_louvain.best_partition(G)
print("Communities:", partition)


plt.figure(figsize=(8, 6))
nx.draw(G, pos, with_labels=True, node_color=list(partition.values()), cmap=plt.cm.Set3, node_size=500, font_size=10)
plt.title("Product Network with Communities")
plt.show()

ModuleNotFoundError: No module named 'pandas'

In [16]:
import pandas as pd


df = pd.read_csv("products_data.csv")


new_data = {
    "Product": ["iPhone 12", "iPhone 12", "iPhone 12",
                "iPhone 13", "iPhone 13", "iPhone 13",
                "iPhone 14", "iPhone 14", "iPhone 14",
                "iPhone 15", "iPhone 15", "iPhone 15",
                "iPhone 15 Pro", "iPhone 15 Pro", "iPhone 15 Pro"],
    "Price": [699, 720, 710,
              799, 820, 810,
              899, 920, 910,
              999, 1020, 1010,
              1199, 1220, 1210],
    "Region": ["North America", "Europe", "Asia",
               "North America", "Europe", "Asia",
               "North America", "Europe", "Asia",
               "North America", "Europe", "Asia",
               "North America", "Europe", "Asia"]
}


new_df = pd.DataFrame(new_data)


df = df.drop(columns=["Price", "Region"], errors="ignore")
merged_df = pd.merge(new_df, df, on="Product", how="left")


merged_df.to_csv("products_data.csv", index=False)
print("Updated Data with Prices by Region:")
print(merged_df)

Updated Data with Prices by Region:
          Product  Price         Region  \
0       iPhone 12    699  North America   
1       iPhone 12    699  North America   
2       iPhone 12    699  North America   
3       iPhone 12    720         Europe   
4       iPhone 12    720         Europe   
5       iPhone 12    720         Europe   
6       iPhone 12    710           Asia   
7       iPhone 12    710           Asia   
8       iPhone 12    710           Asia   
9       iPhone 13    799  North America   
10      iPhone 13    799  North America   
11      iPhone 13    799  North America   
12      iPhone 13    820         Europe   
13      iPhone 13    820         Europe   
14      iPhone 13    820         Europe   
15      iPhone 13    810           Asia   
16      iPhone 13    810           Asia   
17      iPhone 13    810           Asia   
18      iPhone 14    899  North America   
19      iPhone 14    899  North America   
20      iPhone 14    899  North America   
21      iPhone 14 

In [1]:
#Load the main data from products_data.csv and  Create the Heatmap
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


df = pd.read_csv("products_data.csv")
print("Main Data:")
print(df)


pivot_table = df.pivot_table(index="Region", columns="Product", values="Price", aggfunc="mean")
print("Pivot Table for Heatmap:")
print(pivot_table)


plt.figure(figsize=(10, 6))
heatmap_data = pivot_table.values
im = plt.imshow(heatmap_data, cmap="hot", interpolation="nearest")
plt.colorbar(im, label="Price")
plt.xticks(np.arange(len(pivot_table.columns)), pivot_table.columns, rotation=45)
plt.yticks(np.arange(len(pivot_table.index)), pivot_table.index)
for i in range(len(pivot_table.index)):
    for j in range(len(pivot_table.columns)):
        plt.text(j, i, f"{heatmap_data[i, j]:.1f}", ha="center", va="center", color="purple")
plt.title("Heatmap of iPhone Prices by Region")
plt.xlabel("Product")
plt.ylabel("Region")
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'pandas'